# Power analysis attack on ASCON

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import serial.tools.list_ports as port_list
import sys
sys.path.append( '../ASCON_init_python' )
sys.path.append( '../sca_scripts' )

sub_layer_dict = {
    "hw" : "ASCON_HW",
    "lut_ascon" : "SBOX_ASCON",
    "lut_bilgin" : "SBOX_BILGIN",
    "lut_allouzi" : "SBOX_ALLOUZI",
    "lut_lu_4" : "SBOX_LU_4",
    "lut_lu_5" : "SBOX_LU_5",
    "lut_lu_6" : "SBOX_LU_6",
    "lut_lu_7" : "SBOX_LU_7"
}

sub_layer_type = "hw"

ports = list(port_list.comports())
for p in ports:
    print (p)

project_file = "../../build/sca_test/sca_test_CW305_ascon_init_" + sub_layer_dict[sub_layer_type] + "_300000_1.cwp"
bitstream = r"../../hw/fpga/bitstream/cw305_top_ascon_init_" + sub_layer_dict[sub_layer_type] + ".bit"

print(bitstream)


## Picoscope and CW305 initialization

In [ ]:
from pico_api import PS5000aWrapper
from CW305_api import CW305Wrapper

try:
    # Initialize picoscope
    ps = PS5000aWrapper()
    ps.get_unitInfo()
    ps.scope_setup()
    # Initialize CW305
    cw305 = CW305Wrapper(ps, bitstream)

except ModuleNotFoundError as e:
    print(e)

## Online phase : ASCON power traces capture

In [ ]:
from tqdm.notebook import tnrange
import chipwhisperer as cw
from operations_init import ascon_init as ascon

# Number of traces to capture
N = 60000

project_file = "../../build/sca_test/sca_test_CW305_ascon_init_" + sub_layer_dict[sub_layer_type] + "_" + str(N) + ".cwp"
project = cw.create_project(project_file, overwrite=True)

key = 0x000102030405060708090A0B0C0D0E0F
key_str = f"{key:032x}"
key_list = [int(key_str[i:i+2], 16) for i in range(0, len(key_str), 2)]

ktp = cw.ktp.Basic()
kk, nonce = ktp.next()

# Each element of the key is converted to a 2-digit hex string 
print("Key: ", [ hex(subkey) for subkey in key_list])

# Write the key to the CW305
cw305.set_key(key_list)
# Dummy capture call due to bug of using AC coupling
cw305.capture_trace_1_round(project, nonce, dummy=True)
    
for i in tnrange(N, desc='Capturing traces'):
    response = cw305.capture_trace_1_round(project, nonce)

    # Sanity check with expected ciphertext
    state = ascon(key.to_bytes(16,'big'), nonce, "Ascon-128", sub_layer_type)
    assert (list(state) == list(response)), "Incorrect encryption result!\nGot {}\nExp {}\n".format(list(response), list(state))
    kk, nonce = ktp.next()

project.save()
project.close()
# Disconnect CW305 and picoscope
cw305.dis()
ps.dis()
